# Zomi Dictionary Fetcher

Fetches the Zomi dictionary from ZomiDictionary.com using their search interface.

## How It Works
1. **Get word list** from existing sources or manual input
2. **Fetch definitions** for each word using search page
3. **Extract** headword, definition, and examples
4. **Save** results in JSONL format

## Notes
- This fetcher works with the ZomiDictionary.com search interface
- Respects rate limiting with delays between requests
- Supports resumable processing via existing output files

In [ ]:
# Step 0: Setup
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

def check_internet() -> bool:
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

if not check_internet():
    raise SystemError("No internet connection.")
print("Internet: OK")

try:
    import requests  # noqa: F401
    print("Packages: OK")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "-q"])
    print("Packages installed.")

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"IS_KAGGLE: {IS_KAGGLE}  |  WORK_DIR: {WORK_DIR}")

In [ ]:
# Step 1: Prepare word list
# For now, we'll use a simple test word list
# In practice, you would load from existing Zomi word sources

TEST_WORDS = ["taste"]

# Save word list for reference
word_list_path = WORK_DIR / "zomi_search_words.txt"
word_list_path.write_text("\n".join(TEST_WORDS) + "\n", encoding="utf-8")
print(f"Word list saved: {word_list_path}")
print(f"Words to process: {len(TEST_WORDS)}")
print(f"Sample: {TEST_WORDS[:5]}")

# Configuration
MAX_WORKERS = 1  # Keep low to be respectful to the server
TIMEOUT = 20.0
RETRIES = 2
SLEEP_BETWEEN_REQUESTS = 0.5  # seconds

est_minutes = len(TEST_WORDS) * (SLEEP_BETWEEN_REQUESTS + 2) / 60  # Rough estimate
print(f"Estimated time: ~{est_minutes:.1f} minutes")

In [ ]:
# Step 2: Fetcher logic — HTML parser + HTTP
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from html import unescape
from html.parser import HTMLParser
from typing import Any, Dict, List
from urllib.parse import quote

BASE_SEARCH_URL = "https://zomidictionary.com/results.php?query="
NO_RESULT_MARKER = "No results found"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/123.0.0.0 Safari/537.36"
)

OUTPUT_DIR = WORK_DIR / "zomi_batches"
PROGRESS_LOG = WORK_DIR / "zomi_fetch_progress.log"
STATUS_FILE = WORK_DIR / "zomi_session_status.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _heartbeat(msg: str) -> None:
    line = f"{time.strftime('%H:%M:%S')}  {msg}\n"
    sys.stdout.write(line)
    sys.stdout.flush()
    with open(PROGRESS_LOG, "a", encoding="utf-8") as f:
        f.write(line)

def _write_status(status: str, written: int = 0, found: int = 0, errors: int = 0, rate: float = 0.0) -> None:
    try:
        data = json.loads(STATUS_FILE.read_text()) if STATUS_FILE.exists() else {}
        data["single"] = {"status": status, "written": written, "found": found, "errors": errors, "rate": round(rate, 1), "updated": time.strftime("%H:%M:%S")}
        STATUS_FILE.write_text(json.dumps(data, indent=2))
    except Exception:
        pass

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", unescape(text)).strip()

class ZomiResultParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.results: List[Dict[str, Any]] = []
        self._current: Dict[str, Any] | None = None
        self._in_pronoun = False
        self._in_definition = False
        self._in_examples_dl = False
        self._in_example_dt = False
        self._in_example_dd = False
        self._headword = ""
        self._definition_parts: List[str] = []
        self._current_example_parts: List[str] = []
        self._examples: List[str] = []

    def handle_starttag(self, tag: str, attrs: List[tuple[str, str | None]]) -> None:
        attr_map = dict(attrs)
        classes = (attr_map.get("class") or "").split()
        
        # Capture headword from pronunciation element
        if tag == "p" and "pronoun" in classes:
            self._in_pronoun = True
        
        # Start collecting definition text when we enter a blockquote
        if tag == "blockquote":
            if not hasattr(self, '_blockquote_count'):
                self._blockquote_count = 0
            self._blockquote_count += 1
            # First and second blockquotes contain definition
            if self._blockquote_count <= 2:
                self._in_definition = True
                self._definition_parts = []
        
        # Look for examples section
        elif tag == "h3" and "Examples" in str(attrs):
            self._in_examples_section = True
        
        # Look for definition list (dl) within examples section
        elif tag == "dl" and getattr(self, '_in_examples_section', False):
            self._in_examples_dl = True
        
        # Look for definition term (dt) - example labels like "Example #1"
        elif tag == "dt" and self._in_examples_dl:
            self._in_example_dt = True
            self._in_example_dd = False
        
        # Look for description description (dd) - actual example sentences
        elif tag == "dd" and self._in_examples_dl:
            self._in_example_dd = True
            self._in_example_dt = False
            self._current_example_parts = []

    def handle_data(self, data: str) -> None:
        if self._in_pronoun:
            self._headword = normalize_space(data)
            self._in_pronoun = False
        elif self._in_definition:
            self._definition_parts.append(data)
        elif self._in_example_dd:
            self._current_example_parts.append(data)

    def handle_endtag(self, tag: str) -> None:
        if tag == "p" and self._in_pronoun:
            self._in_pronoun = False
        
        if tag == "blockquote":
            if self._in_definition:
                text = normalize_space("".join(self._definition_parts))
                if text and self._current is not None:
                    self._current["definition"] = text
                self._in_definition = False
        
        if tag == "dl" and self._in_examples_dl:
            self._in_examples_dl = False
            if hasattr(self, '_in_examples_section'):
                self._in_examples_section = False
        
        if tag == "dt" and self._in_example_dt:
            self._in_example_dt = False
        
        if tag == "dd" and self._in_example_dd:
            text = normalize_space("".join(self._current_example_parts))
            if text and self._current is not None:
                if "examples" not in self._current:
                    self._current["examples"] = []
                self._current["examples"].append(text)
            self._in_example_dd = False
            self._current_example_parts = []

    def _finalize_result(self) -> Dict[str, Any]:
        # Clean up definition
        definition = normalize_space("".join(self._definition_parts))
        
        # Process examples
        processed_examples = []
        for example in self._examples:
            if example:
                example = normalize_space(example)
                if example:
                    processed_examples.append(example)
        
        return {
            "headword": self._headword,
            "definition": definition,
            "examples": processed_examples,
        }

def parse_results(html_text: str) -> List[Dict[str, Any]]:
    parser = ZomiResultParser()
    parser.feed(html_text)
    parser.close()
    result = parser._finalize_result()
    return [result] if result["headword"] or result["definition"] or result["examples"] else []

_thread_local = threading.local()

def _get_thread_session():
    if not hasattr(_thread_local, "session"):
        import requests as req
        s = req.Session()
        s.headers.update({"User-Agent": USER_AGENT})
        s.mount("https://", req.adapters.HTTPAdapter(
            pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS, pool_block=False
    ))
        _thread_local.session = s
    return _thread_local.session

def lookup_word(word: str) -> Dict[str, Any]:
    source_url = BASE_SEARCH_URL + quote(word, safe="")
    session = _get_thread_session()
    resp = session.get(source_url, timeout=TIMEOUT)
    resp.raise_for_status()
    html_text = resp.text
    if NO_RESULT_MARKER in html_text:
        return {"query": word, "source_url": source_url, "found": False, "result_count": 0, "results": [], "error": None}
    results = parse_results(html_text)
    if results:
        return {
            "query": word,
    "source_url": source_url,
            "found": True,
            "result_count": len(results),
            "results": results,
            "error": None
        }
    else:
        return {
            "query": word,
            "source_url": source_url,
            "found": False,
            "result_count": 0,
            "results": [],
            "error": None
        }

def load_processed_queries(output_path: Path) -> set[str]:
    processed: set[str] = set()
    if not output_path.exists():
        return processed
    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            query = str(obj.get("query", "")).strip()
            if query:
                processed.add(query)
    return processed

_heartbeat("Fetcher functions loaded.")

In [ ]:
# Step 3: Process words

SESSION_OUTPUT = OUTPUT_DIR / "zomi_session.jsonl"

start_time = time.time()
_heartbeat(f"Starting: {len(TEST_WORDS)} words")

try:
    processed = load_processed_queries(SESSION_OUTPUT)
    pending = [w for w in TEST_WORDS if w not in processed]
    skipped = len(TEST_WORDS) - len(pending)
    
    if not pending:
        _heartbeat(f"All {len(TEST_WORDS)} words already processed. Skipping.")
        _write_status("skipped", skipped=skipped)
        summary = {"attempted": 0, "written": 0, "found": 0, "missed": 0, "errors": 0, "elapsed": 0, "skipped": skipped}
    else:
        written = found = missed = errors = 0
        error_types: Dict[str, int] = {}
        write_lock = threading.Lock()
        
        _write_status("running")
        mode = "a" if SESSION_OUTPUT.exists() else "w"
        with SESSION_OUTPUT.open(mode, encoding="utf-8") as out:
            for i, word in enumerate(pending):
                _heartbeat(f"Processing {word} ({i+1}/{len(pending)})")
                try:
                    _heartbeat(f"  Fetching...")
                    record = lookup_word(word)
                    if record["found"]:
                        found += 1
                    else:
                        missed += 1
                    with write_lock:
                        out.write(json.dumps(record, ensure_ascii=False) + "\n")
                        out.flush()
                        written += 1
                    
                    # Be respectful - sleep between requests
                    if i < len(pending) - 1:  # Don't sleep after last item
                        time.sleep(SLEEP_BETWEEN_REQUESTS)
                except Exception as exc:
                    errors += 1
                    error_types[type(exc).__name__] = error_types.get(type(exc).__name__, 0) + 1
                    record = {
                        "query": word,
                        "source_url": BASE_SEARCH_URL + quote(word, safe=""),
                        "found": False,
                        "result_count": 0,
                        "results": [],
                        "error": str(exc)
                    }
                    with write_lock:
                        out.write(json.dumps(record, ensure_ascii=False) + "\n")
                        out.flush()
                        written += 1
                    
                    if i < len(pending) - 1:
                        time.sleep(SLEEP_BETWEEN_REQUESTS)
        
        elapsed = time.time() - start_time
        _write_status("done", written=written, found=found, errors=errors)
        summary = {
            "attempted": len(pending),
            "written": written,
            "found": found,
            "missed": missed,
            "errors": errors,
            "elapsed": round(elapsed, 1),
            "skipped": skipped,
            "error_types": error_types
        }
except Exception as e:
    _heartbeat(f"FAILED: {e}")
    _write_status("failed")
    summary = {"attempted": 0, "written": 0, "found": 0, "missed": 0, "errors": 0, "elapsed": 0, "error_types": {}}

total_elapsed = time.time() - start_time
rate = summary.get("written", 0) / total_elapsed if total_elapsed > 0 else 0
_heartbeat(f"Complete in {total_elapsed/60:.1f} minutes ({rate:.1f} w/s)")

print(f"\n{'='*50}")
print(f"Zomi Fetcher Summary")
print(f"{'='*50}")
print(f"Words processed: {summary.get('written', 0)}")
print(f"Found: {summary.get('found', 0)}")
print(f"Missed: {summary.get('missed', 0)}")
print(f"Errors: {summary.get('errors', 0)}")
print(f"Rate: {rate:.1f} w/s")
print(f"Time: {total_elapsed/60:.1f} minutes")
print(f"Output: {SESSION_OUTPUT}")

In [ ]:
# Step 4: Check results
if SESSION_OUTPUT.exists():
    print(f"Results file: {SESSION_OUTPUT}")
    with open(SESSION_OUTPUT) as f:
        lines = f.readlines()
    print(f"Total lines: {len(lines)}")
    for i, line in enumerate(lines[:3]):  # Show first 3 results
        line = line.strip()
        if line:
            try:
                obj = json.loads(line)
                print(f"\nResult {i+1}:")
                print(f"  Query: {obj.get('query')}")
                print(f"  Found: {obj.get('found')}")
                if obj.get('found'):
                    print(f"  Headword: {obj.get('results', [{}])[0].get('headword', 'N/A')}")
                    print(f"  Definition: {obj.get('results', [{}])[0].get('definition', 'N/A')[:100]}...")
                    examples = obj.get('results', [{}])[0].get('examples', [])
                    if examples:
                        print(f"  Examples: {examples}")
            except json.JSONDecodeError:
                print(f"  Invalid JSON: {line[:50]}...")
else:
    print("No results file found.")

# Next Steps

1. **Expand word list**: Replace `TEST_WORDS` with comprehensive Zomi word lists from:
   - Existing Zoli datasets in `data/`
   - Manual word lists
   - Other sources

2. **Improve parsing**: Enhance the HTML parser to better extract:
   - More precise headword detection
   - Better example sentence extraction
   - Pronunciation/audio information

3. **Add multi-session support**: Like the TongDot fetcher, split work across multiple processes

4. **Error handling**: Improve retry logic and error reporting

5. **Validation**: Add result validation to ensure data quality

---
*Note: Always be respectful when scraping websites. Adjust SLEEP_BETWEEN_REQUESTS as needed.*